# Fish Toxicity Regression
In this dataset, each row is an experiment on the minnows. The columns show what chemicals were present, how long the fish were left in the toxic water, the temperature of the water, etc. With that, our target variable is "result_conc1_mean_mol_log" which is the The base-10 logarithm of the mean molar concentration log10(mol/L) required to reach the specified experimental toxicity endpoint (LC50)

In this version:
- pick features to include based on these categories: Test conditions, molecule description, and chemical properties
- change back into a regression task (idea of first having a regression task, picking a threshold, and classifying whether toxic or non-toxic)
- scaling features

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import PredefinedSplit

# read csv
# the fingerprints must be read as text, otherwise pandas turns the 0/1 strings into huge numbers
fp_text = {col: str for col in ["chem_MACCS_fp", "chem_pcp_fp", "chem_Morgan_fp", "chem_ToxPrint_fp"]}
df = pd.read_csv(
    "../adore_dataset/processed/s-F2F-2_mortality.csv",
    dtype=fp_text,
    low_memory=False,
)

target = "result_conc1_mean_mol_log"

# test conditions (decided before the experiment, so no leakage)
categorical_cols = ["test_exposure_type", "result_conc1_type", "test_media_type"]
ordinal_cols = ["result_obs_duration_mean"]          # 24/48/72/96 h

# chemical properties
numeric_cols = ["chem_mw", "chem_ws", "chem_mp", "chem_rdkit_clogp"]

# molecule description: MACCS fingerprint, one column per bit
maccs = df["chem_MACCS_fp"].apply(lambda s: pd.Series(list(s))).astype(int)
maccs.columns = [f"MACCS_{i:03d}" for i in maccs.columns]

X = pd.concat([df[categorical_cols + ordinal_cols + numeric_cols], maccs], axis=1)
y = df[target]

# train/test split: the predefined split by chemical (occurrence)
train = df["split_occurrence"] != "test"
test = df["split_occurrence"] == "test"

X_train, X_test = X[train], X[test]
y_train, y_test = y[train], y[test]

# keep only bits that vary (not always 1s or not always 0s)
fp_cols = [c for c in maccs.columns if X_train[c].std() >= 0.1]
X_train = X_train[categorical_cols + ordinal_cols + numeric_cols + fp_cols]
X_test = X_test[categorical_cols + ordinal_cols + numeric_cols + fp_cols]

# cross-validation folds 0-4 from the same predefined split
ps = PredefinedSplit(df.loc[train, "split_occurrence"].astype(int).values)

print("train:", X_train.shape, " test:", X_test.shape, " MACCS bits kept:", len(fp_cols))
print("missing values:", X_train.isna().sum().sum(), X_test.isna().sum().sum())


train: (3550, 146)  test: (669, 146)  MACCS bits kept: 138
missing values: 0 0


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler

preprocessing = ColumnTransformer([
    ("scaler", StandardScaler(), numeric_cols),
    ("one_hot", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
    ("duration", MinMaxScaler(), ordinal_cols),       
    ("maccs", "passthrough", fp_cols),               
])


In [4]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_validate
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Elastic Net": ElasticNet(),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

cv_results = []
for name, model in models.items():
    pipeline = make_pipeline(preprocessing, model)
    scores = cross_validate(pipeline, X_train, y_train, cv=ps,
                            scoring={"rmse": "neg_root_mean_squared_error", "r2": "r2"})
    rmse = -scores["test_rmse"]
    cv_results.append({"model": name,
                       "RMSE": rmse.mean(), "RMSE std": rmse.std(),
                       "R2": scores["test_r2"].mean()})
    print(f"{name:18s} RMSE {rmse.mean():.3f} ± {rmse.std():.3f}   R2 {scores['test_r2'].mean():.3f}")

/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Linear Regression  RMSE 1.405 ± 0.311   R2 0.174
Ridge              RMSE 1.265 ± 0.253   R2 0.346


/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Lasso              RMSE 1.530 ± 0.202   R2 0.052
Elastic Net        RMSE 1.307 ± 0.150   R2 0.306


/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


...

Decision Tree      RMSE 1.389 ± 0.179   R2 0.189


/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Random Forest      RMSE 1.014 ± 0.088   R2 0.568


/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Gradient Boosting  RMSE 0.993 ± 0.086   R2 0.581


Grid Tuning

In [5]:
from sklearn.model_selection import GridSearchCV

param_grids = {
    "SVR": {
        "svr__C": [0.1, 1, 10, 100],
        "svr__epsilon": [0.01, 0.1, 0.5],
        "svr__kernel": ["rbf", "linear"]
    },
    "Random Forest": {
        "randomforestregressor__n_estimators": [100, 200],
        "randomforestregressor__max_depth": [None, 10, 20],
        "randomforestregressor__min_samples_split": [2, 5]
    },
    "Gradient Boosting": {
        "gradientboostingregressor__n_estimators": [100, 200],
        "gradientboostingregressor__learning_rate": [0.01, 0.1],
        "gradientboostingregressor__max_depth": [2, 3, 5]
    }
}

In [6]:
grid_results = []
best_models = {}

for name in ["SVR", "Random Forest", "Gradient Boosting"]:

    pipeline = make_pipeline(preprocessing, models[name])

    grid = GridSearchCV(
        pipeline,
        param_grid=param_grids[name],
        cv=ps,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_

    grid_results.append({
        "Model": name,
        "Best RMSE": -grid.best_score_,
        "Best Parameters": grid.best_params_
    })

    print(f"{name}")
    print(f"RMSE: {-grid.best_score_:.3f}")
    print(f"Best parameters: {grid.best_params_}")
    print()

/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be enc

KeyboardInterrupt: 

In [7]:
print(fp_cols[2])

MACCS_021
